# Rewriting the 3-Decomposition Conjecture

This notebook develops and tests a Collatz-style rewriting formulation of the **3-Decomposition Conjecture** for connected cubic graphs.

## Mathematical status

The notebook contains:

1. an exact reformulation using only a spanning tree;
2. a rigorously justified local rewrite, the **endpoint leaf-slide**;
3. exhaustive transition-graph tests on standard cubic graphs;
4. a concrete example proving that leaf-slides alone are insufficient;
5. a stronger computational target, the **Monotone Exchange Conjecture**.

The experiments are evidence, not a proof.

For a connected cubic graph \(G\) and spanning tree \(T\), put

\[
H_T=G-E(T).
\]

Because \(G\) is cubic, \(H_T\) has maximum degree at most two. Let

\[
b(T)=\#\{\text{path components of }H_T\text{ having at least two edges}\}.
\]

Then \(T\) extends to a 3-decomposition exactly when \(b(T)=0\).

In [1]:
# Colab-safe setup
import sys
import subprocess
import importlib.util

required = {
    "networkx": "networkx>=3.2",
    "pandas": "pandas>=2.0",
}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from collections import Counter
from dataclasses import dataclass
import random
import time

import networkx as nx
import pandas as pd

print("NetworkX:", nx.__version__)

NetworkX: 3.6.1


## 1. Core representation

An edge is stored canonically as an ordered pair. A spanning tree is represented by a `frozenset` of canonical edges.

A path component of \(H_T\) with one edge is harmless: it belongs to the matching part. A cycle component is also harmless: it belongs to the cycle part. Only paths with at least two edges are defects.

In [2]:
def edge(u, v):
    """Canonical undirected edge."""
    return (u, v) if u <= v else (v, u)


def graph_edges(G):
    return frozenset(edge(u, v) for u, v in G.edges())


def tree_edges(T_graph):
    return frozenset(edge(u, v) for u, v in T_graph.edges())


def tree_graph(G, T):
    F = nx.Graph()
    F.add_nodes_from(G.nodes())
    F.add_edges_from(T)
    return F


def cotree_graph(G, T):
    H = nx.Graph()
    H.add_nodes_from(G.nodes())
    H.add_edges_from(e for e in graph_edges(G) if e not in T)
    return H


def ordered_path(component):
    """Return the vertices of a nontrivial path component in path order."""
    ends = [v for v, d in component.degree() if d == 1]
    if len(ends) != 2:
        raise ValueError("Component is not a nontrivial path.")
    return nx.shortest_path(component, ends[0], ends[1])


def bad_path_components(G, T):
    """Ordered cotree path components containing at least two edges."""
    H = cotree_graph(G, T)
    bad = []
    for vertices in nx.connected_components(H):
        C = H.subgraph(vertices).copy()
        if C.number_of_edges() < 2:
            continue
        degrees = dict(C.degree())
        if nx.is_tree(C) and max(degrees.values(), default=0) <= 2:
            bad.append(ordered_path(C))
    return bad


def b_value(G, T):
    return len(bad_path_components(G, T))


def defect_edges(G, T):
    """
    Non-tree edges whose endpoint tree-degrees are {1,2}.
    Each bad cotree path has exactly two such end edges.
    """
    F = tree_graph(G, T)
    return frozenset(
        e for e in graph_edges(G) - T
        if sorted((F.degree(e[0]), F.degree(e[1]))) == [1, 2]
    )


def is_spanning_tree(G, T):
    return len(T) == G.number_of_nodes() - 1 and nx.is_tree(tree_graph(G, T))


def decomposition_from_tree(G, T):
    """When b(T)=0, return the tree, cycle edges, and matching edges."""
    if b_value(G, T) != 0:
        raise ValueError("This tree still has bad cotree path components.")

    H = cotree_graph(G, T)
    cycle_edges = set()
    matching_edges = set()

    for vertices in nx.connected_components(H):
        C = H.subgraph(vertices)
        if C.number_of_edges() == 0:
            continue
        if C.number_of_edges() == 1:
            matching_edges.update(edge(u, v) for u, v in C.edges())
        elif all(d == 2 for _, d in C.degree()):
            cycle_edges.update(edge(u, v) for u, v in C.edges())
        else:
            raise AssertionError("Unexpected cotree component when b(T)=0.")

    return {
        "tree": frozenset(T),
        "cycles": frozenset(cycle_edges),
        "matching": frozenset(matching_edges),
    }

## 2. Exact tree-only reformulation

### Lemma

For every connected cubic graph \(G\) and spanning tree \(T\),

\[
T\text{ extends to a 3-decomposition}
\quad\Longleftrightarrow\quad
b(T)=0.
\]

Moreover,

\[
\#\{uv\in E(G)\setminus E(T):
\{\deg_T(u),\deg_T(v)\}=\{1,2\}\}=2b(T).
\]

### Proof

In \(H_T\), a vertex has degree \(3-\deg_T(v)\), hence degree at most two.

* A cycle component has only vertices of tree-degree one.
* A one-edge path has two vertices of tree-degree two.
* A path with at least two edges has exactly two end edges joining a tree-degree-two endpoint to a tree-degree-one internal vertex.

Thus each bad path contributes exactly two defect edges and no other component contributes any. If no bad path exists, the path components form a matching and the cycle components form a disjoint union of cycles.

In [3]:
from networkx.algorithms.tree.mst import SpanningTreeIterator


def all_spanning_trees(G):
    for T_graph in SpanningTreeIterator(G):
        yield tree_edges(T_graph)


def verify_tree_only_lemma(G, max_trees=None):
    checked = 0
    for T in all_spanning_trees(G):
        assert is_spanning_tree(G, T)
        assert len(defect_edges(G, T)) == 2 * b_value(G, T)

        if b_value(G, T) == 0:
            D = decomposition_from_tree(G, T)
            assert D["tree"] | D["cycles"] | D["matching"] == graph_edges(G)
            assert not (D["tree"] & D["cycles"])
            assert not (D["tree"] & D["matching"])
            assert not (D["cycles"] & D["matching"])

        checked += 1
        if max_trees is not None and checked >= max_trees:
            break
    return checked


for name, G in {
    "K4": nx.complete_graph(4),
    "K3,3": nx.complete_bipartite_graph(3, 3),
    "cube": nx.cubical_graph(),
    "Petersen": nx.petersen_graph(),
}.items():
    print(f"{name:8s}: verified on {verify_tree_only_lemma(G):,} spanning trees")

K4      : verified on 16 spanning trees
K3,3    : verified on 81 spanning trees


cube    : verified on 384 spanning trees


Petersen: verified on 2,000 spanning trees


## 3. Rewrite rules

### All basis exchanges

For every non-tree edge \(e=uv\), adding \(e\) creates the fundamental cycle consisting of \(e\) and the unique \(u\)-\(v\) path in \(T\). Removing any tree edge on that path gives another spanning tree.

### Endpoint leaf-slide

Let

\[
P=v_0v_1\cdots v_k,\qquad k\ge2,
\]

be a bad path in \(H_T\). The internal vertex \(v_1\) has cotree-degree two and therefore tree-degree one. Let \(x\) be its unique tree-neighbor. Define

\[
T'=T+v_0v_1-v_1x.
\]

This is a valid basis exchange: it changes the parent of the tree leaf \(v_1\) from \(x\) to \(v_0\). A direct component analysis gives

\[
b(T')\le b(T).
\]

The move slides the endpoint defect, absorbs a harmless one-edge path, merges two bad paths, or closes a bad path into a cycle.

In [4]:
def basis_exchange_neighbors(G, T):
    """All spanning trees obtainable by one fundamental-cycle exchange."""
    F = tree_graph(G, T)
    result = {}

    for e in graph_edges(G) - T:
        u, v = e
        path = nx.shortest_path(F, u, v)
        path_edges = [edge(path[i], path[i + 1]) for i in range(len(path) - 1)]

        for f in path_edges:
            T2 = frozenset((set(T) | {e}) - {f})
            result[T2] = {"add": e, "remove": f}

    return result


def endpoint_leaf_slide_moves(G, T):
    """Directed endpoint leaf-slides, returned as T' -> metadata."""
    F = tree_graph(G, T)
    moves = {}

    for path in bad_path_components(G, T):
        for oriented in (path, list(reversed(path))):
            v0, v1 = oriented[0], oriented[1]
            tree_neighbors = list(F.neighbors(v1))
            if len(tree_neighbors) != 1:
                raise AssertionError("The first internal vertex must be a tree leaf.")

            x = tree_neighbors[0]
            add = edge(v0, v1)
            remove = edge(v1, x)
            T2 = frozenset((set(T) | {add}) - {remove})

            assert is_spanning_tree(G, T2)
            assert b_value(G, T2) <= b_value(G, T)

            moves[T2] = {
                "bad_path": tuple(oriented),
                "add": add,
                "remove": remove,
                "before_b": b_value(G, T),
                "after_b": b_value(G, T2),
            }

    return moves

## 4. Exhaustive finite-state analysis

For a fixed graph, form the graph of all spanning trees.

* A transition is any basis exchange.
* A **neutral transition** preserves \(b\).
* A **strict transition** lowers \(b\).
* A positive-\(b\) neutral component is **closed** if it has no strict exit.

The following stronger statement would imply the 3-Decomposition Conjecture.

### Monotone Exchange Conjecture

For every connected cubic graph \(G\) and every spanning tree \(T\) with \(b(T)>0\), there is a finite sequence

\[
T=T_0,T_1,\ldots,T_m
\]

of basis exchanges such that

\[
b(T_0)=\cdots=b(T_{m-1})>b(T_m).
\]

Equivalently, no positive-\(b\) neutral component is closed.

In [5]:
@dataclass
class StateSpaceReport:
    graph: str
    vertices: int
    spanning_trees: int
    b_distribution: dict
    positive_one_step_local_minima: int
    positive_neutral_components: int
    closed_positive_neutral_components: int
    terminal_positive_leaf_slide_sccs: int
    seconds: float


def analyze_state_space(G, name="graph", analyze_leaf_slides=True):
    start = time.time()

    states = list(all_spanning_trees(G))
    state_set = set(states)
    bvals = {T: b_value(G, T) for T in states}

    neighbors = {}
    local_minima = 0
    neutral = nx.Graph()
    neutral.add_nodes_from(states)

    for T in states:
        N = basis_exchange_neighbors(G, T)
        assert set(N).issubset(state_set)
        neighbors[T] = set(N)

        if bvals[T] > 0 and all(bvals[U] >= bvals[T] for U in N):
            local_minima += 1

        for U in N:
            if bvals[U] == bvals[T]:
                neutral.add_edge(T, U)

    positive_components = 0
    closed_components = 0

    for component in nx.connected_components(neutral):
        level = bvals[next(iter(component))]
        assert all(bvals[T] == level for T in component)
        if level == 0:
            continue

        positive_components += 1
        has_strict_exit = any(
            bvals[U] < level
            for T in component
            for U in neighbors[T]
        )
        if not has_strict_exit:
            closed_components += 1

    terminal_leaf_sccs = 0
    if analyze_leaf_slides:
        D = nx.DiGraph()
        D.add_nodes_from(states)
        for T in states:
            for U in endpoint_leaf_slide_moves(G, T):
                D.add_edge(T, U)

        for component in nx.strongly_connected_components(D):
            if min(bvals[T] for T in component) == 0:
                continue
            terminal = not any(
                U not in component
                for T in component
                for U in D.successors(T)
            )
            if terminal:
                terminal_leaf_sccs += 1

    return StateSpaceReport(
        graph=name,
        vertices=G.number_of_nodes(),
        spanning_trees=len(states),
        b_distribution=dict(sorted(Counter(bvals.values()).items())),
        positive_one_step_local_minima=local_minima,
        positive_neutral_components=positive_components,
        closed_positive_neutral_components=closed_components,
        terminal_positive_leaf_slide_sccs=terminal_leaf_sccs,
        seconds=round(time.time() - start, 3),
    )


canonical_graphs = {
    "K4": nx.complete_graph(4),
    "K3,3": nx.complete_bipartite_graph(3, 3),
    "cube": nx.cubical_graph(),
    "Petersen": nx.petersen_graph(),
}

reports = [
    analyze_state_space(G, name)
    for name, G in canonical_graphs.items()
]

pd.DataFrame([r.__dict__ for r in reports])

,graph,vertices,spanning_trees,b_distribution,positive_one_step_local_minima,positive_neutral_components,closed_positive_neutral_components,terminal_positive_leaf_slide_sccs,seconds
0,K4,4,16,"{0: 4, 1: 12}",0,1,0,0,0.011
1,"K3,3",6,81,"{0: 9, 1: 72}",0,1,0,0,0.072
2,cube,8,384,"{0: 24, 1: 240, 2: 120}",48,2,0,24,0.503
3,Petersen,10,2000,"{0: 70, 1: 1020, 2: 870, 3: 40}",120,42,0,0,3.832


The key distinction in the table is:

* positive one-step local minima may exist because \(b\) can be constant on a plateau;
* no tested canonical graph has a **closed** positive plateau under all non-increasing basis exchanges;
* endpoint leaf-slides alone do have terminal positive strongly connected components on the cube.

Thus a finite auxiliary exchange rule is genuinely necessary.

In [6]:
# A concrete cube state trapped by the restricted leaf-slide dynamics,
# but solved by another non-increasing basis exchange.

G = nx.cubical_graph()
T = frozenset({
    edge(0, 1),
    edge(0, 3),
    edge(1, 7),
    edge(2, 3),
    edge(3, 5),
    edge(4, 5),
    edge(5, 6),
})

print("Initial spanning tree:", sorted(T))
print("Initial bad cotree paths:", bad_path_components(G, T))
print("Initial b(T):", b_value(G, T))

leaf_moves = endpoint_leaf_slide_moves(G, T)
print("\nEndpoint leaf-slides:")
for U, meta in leaf_moves.items():
    print(meta)

# Add cotree edge (4,7), remove tree edge (1,7).
T2 = frozenset((set(T) | {edge(4, 7)}) - {edge(1, 7)})

print("\nAuxiliary exchange: add (4,7), remove (1,7)")
print("New bad cotree paths:", bad_path_components(G, T2))
print("New b(T):", b_value(G, T2))
print(
    "Cotree components:",
    [sorted(C) for C in nx.connected_components(cotree_graph(G, T2))]
)

assert b_value(G, T) == 1
assert b_value(G, T2) == 0

Initial spanning tree: [(0, 1), (0, 3), (1, 7), (2, 3), (3, 5), (4, 5), (5, 6)]
Initial bad cotree paths: [[0, 4, 7, 6, 2, 1]]
Initial b(T): 1

Endpoint leaf-slides:
{'bad_path': (0, 4, 7, 6, 2, 1), 'add': (0, 4), 'remove': (4, 5), 'before_b': 1, 'after_b': 1}
{'bad_path': (1, 2, 6, 7, 4, 0), 'add': (1, 2), 'remove': (2, 3), 'before_b': 1, 'after_b': 1}

Auxiliary exchange: add (4,7), remove (1,7)
New bad cotree paths: []
New b(T): 0
Cotree components: [[0, 4], [1, 2, 6, 7], [3], [5]]


## 5. Reproducible random sweep

The test below samples connected labelled cubic graphs and exhaustively enumerates every spanning tree of each sampled graph. It then searches for a closed positive neutral plateau.

This is not isomorphism-class enumeration. It is a reproducible stress test controlled by the listed random seeds.

In [7]:
def connected_random_cubic_graph(n, seed):
    if n % 2:
        raise ValueError("A cubic graph must have an even number of vertices.")

    rng = random.Random(seed)
    for _ in range(10_000):
        trial_seed = rng.randrange(2**32)
        G = nx.random_regular_graph(3, n, seed=trial_seed)
        if nx.is_connected(G):
            return G
    raise RuntimeError("Failed to generate a connected cubic graph.")


# Moderate defaults for an ordinary Colab run.
random_specs = [
    (8, 104729),
    (8, 130363),
    (8, 155921),
    (10, 196613),
    (10, 262147),
    (10, 314159),
    (12, 524287),
]

random_reports = []
for n, seed in random_specs:
    G = connected_random_cubic_graph(n, seed)
    report = analyze_state_space(
        G,
        name=f"random cubic n={n}, seed={seed}",
        analyze_leaf_slides=False,
    )
    random_reports.append(report)

pd.DataFrame([r.__dict__ for r in random_reports])

,graph,vertices,spanning_trees,b_distribution,positive_one_step_local_minima,positive_neutral_components,closed_positive_neutral_components,terminal_positive_leaf_slide_sccs,seconds
0,"random cubic n=8, seed=104729",8,336,"{0: 18, 1: 188, 2: 130}",50,2,0,0,0.113
1,"random cubic n=8, seed=130363",8,256,"{0: 8, 1: 120, 2: 128}",56,2,0,0,0.081
2,"random cubic n=8, seed=155921",8,336,"{0: 18, 1: 188, 2: 130}",50,2,0,0,0.109
3,"random cubic n=10, seed=196613",10,1224,"{0: 18, 1: 438, 2: 752, 3: 16}",224,10,0,0,0.609
4,"random cubic n=10, seed=262147",10,1080,"{0: 18, 1: 338, 2: 644, 3: 80}",168,3,0,0,0.431
5,"random cubic n=10, seed=314159",10,1160,"{0: 14, 1: 374, 2: 712, 3: 60}",212,3,0,0,0.428
6,"random cubic n=12, seed=524287",12,6435,"{0: 61, 1: 1552, 2: 3804, 3: 1018}",813,3,0,0,4.061


In [8]:
total_trees = sum(r.spanning_trees for r in random_reports)
closed = sum(r.closed_positive_neutral_components for r in random_reports)

print(f"Random graphs tested: {len(random_reports)}")
print(f"Spanning trees exhaustively tested: {total_trees:,}")
print(f"Closed positive neutral components found: {closed}")

assert closed == 0, (
    "A closed positive plateau was found. This is a counterexample "
    "to the Monotone Exchange Conjecture and should be inspected."
)

Random graphs tested: 7
Spanning trees exhaustively tested: 10,827
Closed positive neutral components found: 0


## 6. What has and has not been established

### Established exactly

1. \(b(T)=0\) is equivalent to obtaining a 3-decomposition from \(T\).
2. The number of tree-degree-\(1\)/tree-degree-\(2\) cotree edges equals \(2b(T)\).
3. Every endpoint leaf-slide is a valid basis exchange and never increases \(b\).
4. Endpoint leaf-slides alone are insufficient.
5. The displayed auxiliary cube exchange eliminates its defect immediately.

### Experimental finding

No closed positive plateau under **all** non-increasing basis exchanges was found in the exhaustive state spaces displayed above.

### Main next targets

1. **Counterexample search:** encode a closed positive plateau directly in SAT/SMT.
2. **Rule mining:** extract the smallest radius-labelled configurations responsible for strict exits.
3. **Relative termination:** treat \(b\)-lowering rules as strict and \(b\)-preserving moves as neutral.
4. **Automated ranking:** search for lexicographic or matrix-valued combinations of typed subgraph counts.
5. **Local graph rewriting:** introduce a token that walks along a fundamental tree path, converting a globally stated exchange into finitely many local rewrite steps.

A single hand-chosen scalar potential is unlikely to suffice: neutral plateaux are real, and natural secondary statistics can improve on one example while worsening on another. That is precisely where automated matrix or weighted-subgraph interpretations become relevant.

## 7. Second universal rewrite: tail peeling

Let

\[
P=v_0v_1\cdots v_k,\qquad k\ge3,
\]

be a bad cotree path. Orient it, put \(a=v_{k-2}\), \(b=v_{k-1}\), \(c=v_k\), and let \(x\) be the unique tree-neighbor of the cotree-internal vertex \(a\). Perform

\[
T'=T+ab-ax.
\]

In the new cotree, \(bc\) is a harmless one-edge component. The prefix ending at \(a\) is attached to the old cotree component containing \(x\). Since \(x\) has tree-degree at least two, its old cotree-degree is at most one. Hence the move either shortens the selected bad path, absorbs a harmless \(K_2\), or merges two bad paths. In every case,

\[
b(T')\le b(T).
\]

This separates the programme into a length-normalization layer and a hard \(P_2\)-defect layer.

In [9]:
def tail_peel_moves(G, T):
    """All oriented tail-peel exchanges."""
    F = tree_graph(G, T)
    moves = {}

    for P in bad_path_components(G, T):
        for oriented in (P, list(reversed(P))):
            k = len(oriented) - 1
            if k < 3:
                continue

            a = oriented[k - 2]
            b = oriented[k - 1]
            x_neighbors = list(F.neighbors(a))
            assert len(x_neighbors) == 1
            x = x_neighbors[0]

            add = edge(a, b)
            remove = edge(a, x)
            T2 = frozenset((set(T) | {add}) - {remove})

            assert is_spanning_tree(G, T2)
            assert b_value(G, T2) <= b_value(G, T)

            moves[T2] = {
                "path": tuple(oriented),
                "add": add,
                "remove": remove,
                "before_b": b_value(G, T),
                "after_b": b_value(G, T2),
            }

    return moves


# Verify the rule throughout the four already-enumerated canonical state spaces.
checked_tail_moves = 0
for G in canonical_graphs.values():
    for T in all_spanning_trees(G):
        checked_tail_moves += len(tail_peel_moves(G, T))

print(f"Verified tail-peel rewrites on canonical graphs: {checked_tail_moves:,}")

Verified tail-peel rewrites on canonical graphs: 4,440


## 8. The clean fundamental-edge lemma for a \(P_2\) defect

Suppose the cotree contains a bad path \(a-b-c\). Choose \(e=ab\). Removing \(e\) from the cotree leaves the harmless edge \(bc\).

Let \(f=uv\) be a tree edge on the fundamental tree path of \(e\). If both \(u\) and \(v\) are isolated in \(H_T-e\), then

\[
T'=T+e-f
\]

replaces the \(P_2\) by the harmless matching edges \(bc\) and \(uv\). Therefore

\[
b(T')=b(T)-1.
\]

The obstruction is now concrete: every candidate fundamental path must be covered by vertices already occupied by cotree paths or cycles.

In [10]:
def clean_p2_exit_moves(G, T):
    """Strict exchanges supplied by the clean fundamental-edge lemma."""
    H = cotree_graph(G, T)
    F = tree_graph(G, T)
    result = {}

    for P in bad_path_components(G, T):
        if len(P) != 3:
            continue

        a, b, c = P
        for endpoint, center in ((a, b), (c, b)):
            e = edge(endpoint, center)
            H_minus = H.copy()
            H_minus.remove_edge(*e)

            tree_path = nx.shortest_path(F, endpoint, center)
            for i in range(len(tree_path) - 1):
                f = edge(tree_path[i], tree_path[i + 1])
                if H_minus.degree(f[0]) or H_minus.degree(f[1]):
                    continue

                T2 = frozenset((set(T) | {e}) - {f})
                assert is_spanning_tree(G, T2)
                assert b_value(G, T2) == b_value(G, T) - 1
                result[T2] = {"defect": tuple(P), "add": e, "remove": f}

    return result


# Concrete clean exit in one eight-vertex cubic graph.
G_clean = nx.from_graph6_bytes(b"GqCilO")
T_clean = frozenset({
    edge(0, 1), edge(1, 3), edge(2, 5), edge(3, 4),
    edge(4, 5), edge(4, 7), edge(5, 6),
})

print("Bad path before:", bad_path_components(G_clean, T_clean))
clean_moves = clean_p2_exit_moves(G_clean, T_clean)
print("Number of clean exits:", len(clean_moves))
for T2, metadata in clean_moves.items():
    print(metadata, "-> b =", b_value(G_clean, T2))

Bad path before: [[1, 6, 3]]
Number of clean exits: 3
{'defect': (1, 6, 3), 'add': (1, 6), 'remove': (4, 5)} -> b = 0
{'defect': (1, 6, 3), 'add': (3, 6), 'remove': (3, 4)} -> b = 0
{'defect': (1, 6, 3), 'add': (3, 6), 'remove': (4, 5)} -> b = 0


## 9. Stronger exhaustive test through 10 vertices

There are \(1,2,5,19\) connected simple cubic graph isomorphism classes on \(4,6,8,10\) vertices. The fixed graph6 strings below give one representative of each of the 27 classes.

For every representative, the code exhaustively enumerates every spanning tree and every fundamental-cycle basis exchange. It asks whether a positive level component of the defect statistic \(b\) is closed under all \(b\)-preserving exchanges.

This tests the **Monotone Exchange Conjecture**, which is stronger than merely asking whether each graph has one successful spanning tree.

In [11]:
graph6_representatives = {
    4: ["C~"],
    6: ["EZfG", "E]NG"],
    8: ["GqCilO", "GU`_zO", "GsSJHg", "GHqouC", "GLdBKW"],
    10: [
        "IAiKJ@PK_", "IG`iSaPK_", "IOKcmHOQG", "IGdV@a@BG",
        "IIEcJOIcG", "ICSlADo`O", "IKt@?gb`_", "IPPEGgXi?",
        "IlCCAKMJ?", "IqGSWoDWG", "ISHCq`DJ?", "ICWQZAccO",
        "I@oJAiad?", "IPPIDaQB_", "ILFGR?B_g", "IrAIH?XAo",
        "I?CwvBAw?", "IBYJ?IBcO", "IM@GpEg`O",
    ],
}

all_class_reports = []
for n, encodings in graph6_representatives.items():
    graphs = [nx.from_graph6_bytes(s.encode()) for s in encodings]

    # Internal integrity check.
    for i, G in enumerate(graphs):
        assert nx.is_connected(G)
        assert set(dict(G.degree()).values()) == {3}
        for H in graphs[:i]:
            assert not nx.is_isomorphic(G, H)

    for i, G in enumerate(graphs, start=1):
        all_class_reports.append(
            analyze_state_space(
                G,
                name=f"n={n}, class={i}",
                analyze_leaf_slides=False,
            )
        )

all_class_summary = (
    pd.DataFrame([r.__dict__ for r in all_class_reports])
    .groupby("vertices", as_index=False)
    .agg(
        isomorphism_classes=("graph", "count"),
        spanning_tree_states=("spanning_trees", "sum"),
        one_step_local_minima=(
            "positive_one_step_local_minima",
            "sum",
        ),
        positive_neutral_components=(
            "positive_neutral_components",
            "sum",
        ),
        closed_positive_plateaux=(
            "closed_positive_neutral_components",
            "sum",
        ),
        seconds=("seconds", "sum"),
    )
)

all_class_summary

,vertices,isomorphism_classes,spanning_tree_states,one_step_local_minima,positive_neutral_components,closed_positive_plateaux,seconds
0,4,1,16,0,1,0,0.003
1,6,2,156,0,9,0,0.046
2,8,5,1731,226,10,0,0.569
3,10,19,28716,4240,160,0,13.703


In [12]:
print("Isomorphism classes:", len(all_class_reports))
print(
    "Spanning-tree states:",
    f"{sum(r.spanning_trees for r in all_class_reports):,}",
)
print(
    "Closed positive plateaux:",
    sum(
        r.closed_positive_neutral_components
        for r in all_class_reports
    ),
)

assert len(all_class_reports) == 27
assert sum(r.spanning_trees for r in all_class_reports) == 30_619
assert all(
    r.closed_positive_neutral_components == 0
    for r in all_class_reports
)

Isomorphism classes: 27
Spanning-tree states: 30,619
Closed positive plateaux: 0


## 10. Refined termination architecture

The emerging system has three layers.

1. **Length normalization:** endpoint leaf-slides and tail-peels manipulate long cotree paths without increasing \(b\).
2. **Neutral \(P_2\) transport:** a two-edge defect is moved through the tree-cotree structure by basis exchanges.
3. **Strict exits:** a collision, cycle-closing split, or clean fundamental edge lowers \(b\).

The next theorem-sized target is:

> Every terminal neutral \(P_2\)-transport component contains a clean edge, a collision, or a cycle-closing configuration.

The computational counterpart is to mine minimal radius-labelled witnesses for every strict exit and ask an SMT solver to synthesize a finite complete rule family or a matrix-valued ranking for the remaining neutral rules.

## References motivating the programme

* Emre Yolcu, Scott Aaronson, and Marijn J. H. Heule, **“An Automated Approach to the Collatz Conjecture.”**
* Christopher Bachtler and Irene Heinrich, **“A Reduction of the 3-Decomposition Conjecture.”**
* Rui Li, Yezhou Wu, Junchi Yan, and Donglei Yang, **“A SAT Approach to the 3-Decomposition Conjecture.”**
* Guiying Yan, Baoyindureng Wu, and Qinglin Yu, **“An equivalent formulation of the 3-Decomposition Conjecture.”**
* Recent work of Fan, Guo, and Zhou on spanning-tree/cycle/path decompositions with bounded numbers of two-edge path defects.
* Recent graph-transformation work on relative termination by weighted type graphs and weighted subgraph counting.